In [ ]:
import numpy
import pandas
import uproot

import matplotlib
import matplotlib.pyplot as plt
plt.style.use('../mystyle.mplstyle')

import sbruceana

In [ ]:
PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/cc1e0pi/nue-only/"
FILE_NUE_VAR1 = "CNAF_NuE_1eNp0pi_NuMI_var1_NoCut.root"
FILE_NUE_VAR2 = "CNAF_NuE_1eNp0pi_NuMI_var2_NoCut.root"
FILE_NUE_VAR3 = "CNAF_NuE_1eNp0pi_NuMI_var3_NoCut.root"
FILE_NUE_VAR4 = "CNAF_NuE_1eNp0pi_NuMI_var4_NoCut.root"
FILE_NUE_VAR5 = "CNAF_NuE_1eNp0pi_NuMI_var5_NoCut.root"
FILE_NUE_VAR6 = "CNAF_NuE_1eNp0pi_NuMI_var6_NoCut.root"
FILE_NUE_VAR7 = "CNAF_NuE_1eNp0pi_NuMI_var7_NoCut.root"
FILE_NUE_VAR8 = "CNAF_NuE_1eNp0pi_NuMI_var8_NoCut.root"
FILE_NUE_VAR9 = "CNAF_NuE_1eNp0pi_NuMI_var9_fixed_NoCut.root"
FILE_NUE_VAR10 = "CNAF_NuE_1eNp0pi_NuMI_var10_NoCut.root"
FILE_NUE_VAR11 = "CNAF_NuE_1eNp0pi_NuMI_var11_NoCut.root"
FILE_NUE_VAR12 = "CNAF_NuE_1eNp0pi_NuMI_var12_NoCut.root"
FILE_NUE_VAR13 = "CNAF_NuE_1eNp0pi_NuMI_var13_NoCut.root"
FILE_NUE_VAR14 = "CNAF_NuE_1eNp0pi_NuMI_var14_NoCut.root"
FILE_NUE_VAR15 = "CNAF_NuE_1eNp0pi_NuMI_var15_NoCut.root"

DET_VARS = {
  'CV'            : FILE_NUE_VAR12,
  'hi coh noise'  : FILE_NUE_VAR1,
  'hi int noise'  : FILE_NUE_VAR2,
  'recomb'        : FILE_NUE_VAR3,
  'diff'          : FILE_NUE_VAR4,
  'null'          : FILE_NUE_VAR5,
  'hi gain'       : FILE_NUE_VAR6,
  'lo gain'       : FILE_NUE_VAR7,
  'lo int noise'  : FILE_NUE_VAR8,
  'hi lifetime'   : FILE_NUE_VAR9,
  'light'         : FILE_NUE_VAR13,
  'opaque'        : FILE_NUE_VAR14,
  'transp'        : FILE_NUE_VAR15,
}

ALL_TAGS = numpy.array(list(DET_VARS.keys()), dtype=str)

# BINNING = numpy.array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1., 1.1, 1.2, 1.3, 1.4, 1.5, 2, 2.5, 3])

# for detector systematics, we'll use 200 MeV and then extrapolate
BINNING = numpy.array([0.3, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5, 2, 2.5, 3])

#### $\nu_e$-only

In [ ]:
dfs, pots, lives, tags = [], [], [], []

for tag, file in DET_VARS.items():

  # get sbruce tree
  df = sbruceana.io.convert_tree_to_df(
    f"{PATH_TO_SBRUCE}{file}",
    "events/selectedNu"  
  )
  dfs.append(df)

  # get POTs
  pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{file}")
  pots.append(pot)

  # get livetimes
  live = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{file}")
  lives.append(live)

  # get variation tag
  tags.append(tag)

dfs = numpy.array(dfs, dtype=object)
pots = numpy.array(pots)
lives = numpy.array(lives)
tags = numpy.array(tags)

In [ ]:
pots, lives

In [ ]:
fig, ax = plt.subplots(figsize=(4,3), layout='constrained')

var = "recoE"

# width = 0.2; bins = numpy.arange(0.2, 3+width, width)
bins = BINNING

ax = sbruceana.plotting.plot_var_by_category(ax, dfs[0], bins, var, 'nue', True, hatch='//')

for df, pot, tag in zip(dfs[1:], pots[1:], tags[1:]):
  ax = sbruceana.plotting.plot_var(ax, df, bins, var, pots[0] / pot, histtype='step', label=tag)

# gfx
ax.set(
  title  = f'CV vs variations',
  xlabel = f'{var} [GeV]',
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(ncol=2, title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(11)
fig.suptitle('No selection')

fig.savefig("plots/CNAF_NuE_1eNp0pi_NuMI_DetVars.png", dpi=300)

In [ ]:
fig, (ax, ax_ratio) = plt.subplots(
  2, 1,
  figsize     = (5, 4),
  gridspec_kw = {
    'height_ratios': [2., 1], 
    'hspace': 0.025
  },
  sharex      = True,
  layout      = 'constrained'
)

var = "recoE"
bins = BINNING
x = 0.5 * (bins[:-1] + bins[1:])

# variations
DET_VAR_IDX = [11]
DET_VAR_TAG = 'opaque'
i = 0
chisqs = []

for df, pot, tag in zip(dfs[DET_VAR_IDX], pots[DET_VAR_IDX], tags[DET_VAR_IDX]):

  # get common events between CV and variation
  # *before* the event selection 
  cv_common, variation_common = sbruceana.io.get_dfs_overlap(
    dfs[0], df, 
    ['Run', 'Subrun', 'Evt'], 
    common = True
  )

  # then, select the 1eNp0π interactions
  cv_common = cv_common[cv_common.selected == 1]
  variation_common = variation_common[variation_common.selected == 1]

  # plot CV once
  if i == 0:
    ax = sbruceana.plotting.plot_var_by_category(ax, cv_common, bins, var)

  # plot variation
  if len(DET_VAR_IDX) == 1:
    ax = sbruceana.plotting.plot_var(ax, variation_common, bins, var, pots[0] / pot, color='C3', label=tag, histtype='step', linewidth=2)
  else:
    ax = sbruceana.plotting.plot_var(ax, variation_common, bins, var, pots[0] / pot, label=tag, histtype='step', linewidth=2)

  # residuals
  r, er = sbruceana.utils.get_ratio_of_vars(cv_common[var], pots[0], variation_common[var], pot, bins)

  # print chi-squared
  chi2, ndof = sbruceana.utils.chi2_var(
      cv_common, variation_common,
      bins, var,
      weight_cv = 1.0,
      weight_var = pots[0]/pot,
  )
  chisqs.append(chi2)

  if tag == 'null':
    ax_ratio.plot(x, r, color=f'C{i+2}', linewidth=1.25)
    ax_ratio.fill_between(x, r - er, r + er, color=f'C{i+2}', alpha=0.2, ec=None)
  else:
    if len(DET_VAR_IDX) == 1:
      this_color = f'C3'
    else:
      this_color = f'C{i+2}'
    ax_ratio.errorbar(x, r, xerr=numpy.diff(bins)/2, yerr=er, fmt=this_color, ls='', marker='', markersize=6, linewidth=1.5)
  i += 1

# references
ax_ratio.axhline(1, c='black', lw=0.75)
cv,  _ = numpy.histogram(cv_common[var], bins=bins)
cv_err_ratio = numpy.where(cv > 0, numpy.sqrt(cv) / cv, numpy.nan)
ax_ratio.bar(x, 2*cv_err_ratio, width = numpy.diff(bins), bottom = 1-cv_err_ratio, color='gray', alpha=0.3, fill=True, lw=0)

# gfx
ax.set(
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
if len(DET_VAR_IDX) == 1:
  ax.set_title(f'CV vs {DET_VAR_TAG} variation\n$\\chi^2$/ndof={chi2:.1f}/{ndof}')
else:
  ax.set_title(f'CV vs {DET_VAR_TAG} variation\n$\\chi_+^2$/ndof={chisqs[0]:.1f}/{ndof}, $\\chi_-^2$/ndof={chisqs[1]:.1f}/{ndof}')
leg = ax.legend(ncols=2, loc='upper right', fontsize=9.5, title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(10)
ax_ratio.set(
  xlabel = 'reconstructed energy [GeV]',
  ylabel = 'var / CV',
  ylim   = (0.8, 1.2),
)

PLOT_NAME = f"plots/detector/CNAF_NuE_1eNp0pi_NuMI_DetVars_WithRes_{DET_VAR_TAG.replace(' ', '_')}"
fig.savefig(f"{PLOT_NAME}.png", dpi=300); fig.savefig(f"{PLOT_NAME}.pdf", dpi=300)

In [ ]:
fig, (ax, ax_ratio) = plt.subplots(
  2, 1,
  figsize     = (5, 4),
  gridspec_kw = {
    'height_ratios': [2., 1], 
    'hspace': 0.025
  },
  sharex      = True,
  layout      = 'constrained'
)

var = "recoE"
bins = BINNING
x = 0.5 * (bins[:-1] + bins[1:])
NULL_IDX = 5

# variations
DET_VAR_IDX = [11]
DET_VAR_TAG = 'opaque'
i = 0
chisqs = []

for df, pot, tag in zip(dfs[DET_VAR_IDX], pots[DET_VAR_IDX], tags[DET_VAR_IDX]):

  # get common events between CV and variation
  # *before* the event selection 
  cv_common, variation_common = sbruceana.io.get_dfs_overlap(
    dfs[0], df, 
    ['Run', 'Subrun', 'Evt'], 
    common = True
  )

  # then, select the 1eNp0π interactions
  cv_common = cv_common[cv_common.selected == 1]
  variation_common = variation_common[variation_common.selected == 1]

  # also introduce the null variation
  _, null_common = sbruceana.io.get_dfs_overlap(
    dfs[0], dfs[NULL_IDX], 
    ['Run', 'Subrun', 'Evt'], 
    common = True
  )
  null_common = null_common[null_common.selected == 1]

  # plot CV and null once
  if i == 0:
    ax = sbruceana.plotting.plot_var_by_category(ax, cv_common, bins, var)
    ax = sbruceana.plotting.plot_var(ax, null_common, bins, var, pots[0] / pots[NULL_IDX], color='black', label='null', histtype='step', linewidth=1.5, linestyle='--')

  # plot variation
  if len(DET_VAR_IDX) == 1:
    ax = sbruceana.plotting.plot_var(ax, variation_common, bins, var, pots[0] / pot, color='C3', label=tag, histtype='step', linewidth=2)
  else:
    ax = sbruceana.plotting.plot_var(ax, variation_common, bins, var, pots[0] / pot, label=tag, histtype='step', linewidth=2)

  # null variation
  if i == 0:
    r, er = sbruceana.utils.get_ratio_of_vars(cv_common[var], pots[0], null_common[var], pots[5], bins)
    ax_ratio.errorbar(x, r, xerr=numpy.diff(bins)/2, yerr=er, fmt='black', ls='', marker='', markersize=6, linewidth=1.25)

  # residuals
  r, er = sbruceana.utils.get_ratio_of_vars(cv_common[var], pots[0], variation_common[var], pot, bins)

  # print chi-squared
  chi2, ndof = sbruceana.utils.chi2_var(
      cv_common, variation_common,
      bins, var,
      weight_cv = 1.0,
      weight_var = pots[0]/pot,
  )
  chisqs.append(chi2)

  if len(DET_VAR_IDX) == 1:
    this_color = f'C3'
  else:
    this_color = f'C{i+2}'
  ax_ratio.errorbar(x, r, xerr=numpy.diff(bins)/2, yerr=er, fmt=this_color, ls='', marker='', markersize=6, linewidth=1.75)
  
  i += 1

# references
ax_ratio.axhline(1, c='black', lw=0.75)
cv,  _ = numpy.histogram(cv_common[var], bins=bins)
cv_err_ratio = numpy.where(cv > 0, numpy.sqrt(cv) / cv, numpy.nan)
ax_ratio.bar(x, 2*cv_err_ratio, width = numpy.diff(bins), bottom = 1-cv_err_ratio, color='gray', alpha=0.3, fill=True, lw=0)

# gfx
ax.set(
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
if len(DET_VAR_IDX) == 1:
  ax.set_title(f'CV vs {DET_VAR_TAG} variation\n$\\chi^2$/ndof={chi2:.1f}/{ndof}')
else:
  ax.set_title(f'CV vs {DET_VAR_TAG} variation\n$\\chi_+^2$/ndof={chisqs[0]:.1f}/{ndof}, $\\chi_-^2$/ndof={chisqs[1]:.1f}/{ndof}')
leg = ax.legend(ncols=2, loc='upper right', fontsize=9.5, title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(10)
ax_ratio.set(
  xlabel = 'reconstructed energy [GeV]',
  ylabel = 'var / CV',
  ylim   = (0.8, 1.2),
)

PLOT_NAME = f"plots/detector/CNAF_NuE_1eNp0pi_NuMI_DetVars_WithRes_{DET_VAR_TAG.replace(' ', '_')}"
fig.savefig(f"{PLOT_NAME}.png", dpi=300); fig.savefig(f"{PLOT_NAME}.pdf", dpi=300)

In [ ]:
fig, (ax, ax_ratio) = plt.subplots(
  2, 1,
  figsize     = (5, 4),
  gridspec_kw = {
    'height_ratios': [2., 1], 
    'hspace': 0.025
  },
  sharex      = True,
  layout      = 'constrained'
)

var = "recoE"
bins = BINNING
x = 0.5 * (bins[:-1] + bins[1:])
NULL_IDX = 5
null = dfs[NULL_IDX]

# variations
DET_VAR_IDX = [11, 12]
DET_VAR_TAG = 'transparency'
i = 0
chisqs = []

for df, pot, tag in zip(dfs[DET_VAR_IDX], pots[DET_VAR_IDX], tags[DET_VAR_IDX]):

  variation = df[df.selected == 1]
  cv = dfs[0][dfs[0].selected == 1]
  null = null[null.selected == 1]

  # plot CV and null once
  if i == 0:
    ax = sbruceana.plotting.plot_var_by_category(ax, cv, bins, var)
    ax = sbruceana.plotting.plot_var(ax, null, bins, var, pots[0] / pots[NULL_IDX], color='black', label='null', histtype='step', linewidth=1.5, linestyle='--')

  # plot variation
  if len(DET_VAR_IDX) == 1:
    ax = sbruceana.plotting.plot_var(ax, variation, bins, var, pots[0] / pot, color='C3', label=tag, histtype='step', linewidth=2)
  else:
    ax = sbruceana.plotting.plot_var(ax, variation, bins, var, pots[0] / pot, label=tag, histtype='step', linewidth=2)

  # null variation
  if i == 0:
    r, er = sbruceana.utils.get_ratio_of_vars(cv[var], pots[0], null[var], pots[5], bins)
    ax_ratio.errorbar(x, r, xerr=numpy.diff(bins)/2, yerr=er, fmt='black', ls='', marker='', markersize=6, linewidth=1.25)

  # residuals
  r, er = sbruceana.utils.get_ratio_of_vars(cv[var], pots[0], variation[var], pot, bins)

  # print chi-squared
  chi2, ndof = sbruceana.utils.chi2_var(
      cv, variation[variation.selected == 1],
      bins, var,
      weight_cv = 1.0,
      weight_var = pots[0]/pot,
  )
  chisqs.append(chi2)

  if len(DET_VAR_IDX) == 1:
    this_color = f'C3'
  else:
    this_color = f'C{i+2}'
  ax_ratio.errorbar(x, r, xerr=numpy.diff(bins)/2, yerr=er, fmt=this_color, ls='', marker='', markersize=6, linewidth=1.75)
  
  i += 1

# references
ax_ratio.axhline(1, c='black', lw=0.75)
cv,  _ = numpy.histogram(cv[var], bins=bins)
cv_err_ratio = numpy.where(cv > 0, numpy.sqrt(cv) / cv, numpy.nan)
ax_ratio.bar(x, 2*cv_err_ratio, width = numpy.diff(bins), bottom = 1-cv_err_ratio, color='gray', alpha=0.3, fill=True, lw=0)

# gfx
ax.set(
  ylabel = 'slices [#]',
  xlim   = (bins[0], bins[-1]),
)
if len(DET_VAR_IDX) == 1:
  ax.set_title(f'CV vs {DET_VAR_TAG} variation\n$\\chi^2$/ndof={chi2:.1f}/{ndof}')
else:
  ax.set_title(f'CV vs {DET_VAR_TAG} variation\n$\\chi_+^2$/ndof={chisqs[0]:.1f}/{ndof}, $\\chi_-^2$/ndof={chisqs[1]:.1f}/{ndof}')
leg = ax.legend(ncols=2, loc='upper right', fontsize=9.5, title=f'{pots[0]:.2e} POT'); leg.get_title().set_fontsize(10)
ax_ratio.set(
  xlabel = 'reconstructed energy [GeV]',
  ylabel = 'var / CV',
  ylim   = (0.8, 1.2),
)

PLOT_NAME = f"plots/detector/CNAF_NuE_1eNp0pi_NuMI_DetVars_NoCVMatching_WithRes_{DET_VAR_TAG.replace(' ', '_')}"
fig.savefig(f"{PLOT_NAME}.png", dpi=300); fig.savefig(f"{PLOT_NAME}.pdf", dpi=300)